# Workflows ResIPy e galeria das inversões ERT

Este notebook documenta e reproduz os workflows de inversão individual e combinada, com malha quadrilateral, `Cut Corners` e ensaio sintético de cavidade. As 49 figuras finais são referenciadas por caminhos relativos e aparecem diretamente na visualização do notebook no GitHub.

**Estado dos resultados apresentados:** concluídos e publicados na branch `main`.

## Parâmetros usados

- Malha quadrilateral: FMD = 120 m, `elemx=4`, `xgf=1.5`, `zf=1.1`, `zgf=1.5`, `pad=8`.
- Inversão L2 suave e isotrópica: máximo de 10 iterações, tolerância de 1%, piso de erro de 1%.
- Resistividade permitida: 1 a 1.000.000 Ω·m.
- Corte de contato: `abs(cR) > 40.000 Ω`.
- Sintético: fundo = 5.000 Ω·m, cavidade = 50.000 Ω·m, ruído gaussiano = 3%.
- Polo–dipolo: eletrodo 67 marcado como remoto/infinito (`remote=True`, `x=99999`).
- Todas as seções invertidas usam `clipCorners=True`.

In [ ]:
from pathlib import Path
from importlib.metadata import version
import json
import subprocess
import sys

ROOT = Path(r"D:\\WILLIAM\\DOLINA-ELETRICA")
SCRIPTS = ROOT / "06_SCRIPTS_PROCESSAMENTO"
INDIVIDUAL_SCRIPT = SCRIPTS / "resipy_individual_quad_synthetic.py"
COMBINED_SCRIPT = SCRIPTS / "resipy_combined_quad_synthetic.py"

INDIVIDUAL_OUTPUT = ROOT / "05_RESULTADOS_RESIPY" / "INVERSOES_INDIVIDUAIS_QUAD_NOVA_EXECUCAO"
COMBINED_OUTPUT = ROOT / "05_RESULTADOS_RESIPY" / "INVERSOES_COMBINADAS_QUAD_NOVA_EXECUCAO"
CONTACT_MAX_OHM = 40_000.0
PUBLISH_EACH_IMAGE = False  # True: commit e push após cada PNG

assert ROOT.exists(), ROOT
assert INDIVIDUAL_SCRIPT.exists(), INDIVIDUAL_SCRIPT
assert COMBINED_SCRIPT.exists(), COMBINED_SCRIPT
assert version("resipy") == "3.6.6", version("resipy")
print("Python:", sys.executable)
print("ResIPy:", version("resipy"))

## Fluxo de processamento

```text
BIN filtrado + topografia
        │
        ├── importação Syscal e QC (contato ≤ 40 kΩ)
        ├── pseudoseção observada
        ├── malha quadrilateral
        ├── inversão real R2 → seção Cut Corners
        └── quadrupolos reais após QC
                    │
modelo paramétrico ─┼── forward sintético + 3% de ruído
                    ├── reinversão desde fundo homogêneo
                    └── métricas e comparação verdadeiro/sintético/real
```

No combinado, DD+WS são reunidos em L1; DD+WS+PD são reunidos em L2. O PD usa o eletrodo 67 no infinito.

## Executar os cinco arranjos individualmente

A pasta de saída deve ser nova. O script não sobrescreve resultados existentes.

In [ ]:
if INDIVIDUAL_OUTPUT.exists():
    raise FileExistsError(f"Escolha outra saída: {INDIVIDUAL_OUTPUT}")

individual_command = [
    sys.executable, str(INDIVIDUAL_SCRIPT), str(ROOT), str(INDIVIDUAL_OUTPUT),
    "--contact-max-ohm", str(CONTACT_MAX_OHM),
    "--cases", "L1_DD", "L1_WS", "L2_DD", "L2_WS", "L2_PD",
]
if PUBLISH_EACH_IMAGE:
    individual_command.append("--publish-images")
subprocess.run(individual_command, check=True)

## Executar os arranjos combinados

Executa L1 DD+WS e L2 DD+WS+PD com os mesmos parâmetros dos individuais.

In [ ]:
if COMBINED_OUTPUT.exists():
    raise FileExistsError(f"Escolha outra saída: {COMBINED_OUTPUT}")

combined_command = [
    sys.executable, str(COMBINED_SCRIPT), str(ROOT), str(COMBINED_OUTPUT),
    "--contact-max-ohm", str(CONTACT_MAX_OHM),
]
if PUBLISH_EACH_IMAGE:
    combined_command.append("--publish-images")
subprocess.run(combined_command, check=True)

## Validar o eletrodo infinito após a execução

A validação deve indicar um único remoto de rótulo 67.

In [ ]:
remote_checks = list(COMBINED_OUTPUT.rglob("validacao_eletrodo_infinito.json"))
for path in remote_checks:
    validation = json.loads(path.read_text(encoding="utf-8"))
    print(path.relative_to(COMBINED_OUTPUT))
    print(json.dumps(validation, indent=2, ensure_ascii=False))
    assert validation["valid"] is True

# Galeria dos resultados publicados

As imagens abaixo são os arquivos versionados no repositório. Não dependem da execução das células para aparecer no GitHub.

## L1-DD individual

### Pseudoseção observada
![L1-DD pseudoseção](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_2026-10-30/L1_DD/REAL/01_pseudosecao_observada.png)

### Malha quadrilateral
![L1-DD malha](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_2026-10-30/L1_DD/REAL/02_malha_quadrilateral.png)

### Modelo real — Cut Corners
![L1-DD real](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_2026-10-30/L1_DD/REAL/03_modelo_resistividade_cut_corners.png)

### Modelo sintético verdadeiro
![L1-DD verdadeiro](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_2026-10-30/L1_DD/SINTETICO/01_modelo_verdadeiro_cut_corners.png)

### Pseudoseção sintética
![L1-DD pseudoseção sintética](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_2026-10-30/L1_DD/SINTETICO/02_pseudosecao_sintetica.png)

### Recuperação sintética
![L1-DD recuperação](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_2026-10-30/L1_DD/SINTETICO/03_recuperacao_sintetica_cut_corners.png)

### Comparação
![L1-DD comparação](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_2026-10-30/L1_DD/COMPARACAO_VERDADEIRO_SINTETICO_REAL_CUT_CORNERS.png)

## L1-WS individual

### Pseudoseção observada
![L1-WS pseudoseção](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_CONTINUACAO_2026-10-30/L1_WS/REAL/01_pseudosecao_observada.png)

### Malha quadrilateral
![L1-WS malha](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_CONTINUACAO_2026-10-30/L1_WS/REAL/02_malha_quadrilateral.png)

### Modelo real — Cut Corners
![L1-WS real](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_CONTINUACAO_2026-10-30/L1_WS/REAL/03_modelo_resistividade_cut_corners.png)

### Modelo sintético verdadeiro
![L1-WS verdadeiro](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_CONTINUACAO_2026-10-30/L1_WS/SINTETICO/01_modelo_verdadeiro_cut_corners.png)

### Pseudoseção sintética
![L1-WS pseudoseção sintética](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_CONTINUACAO_2026-10-30/L1_WS/SINTETICO/02_pseudosecao_sintetica.png)

### Recuperação sintética
![L1-WS recuperação](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_CONTINUACAO_2026-10-30/L1_WS/SINTETICO/03_recuperacao_sintetica_cut_corners.png)

### Comparação
![L1-WS comparação](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_CONTINUACAO_2026-10-30/L1_WS/COMPARACAO_VERDADEIRO_SINTETICO_REAL_CUT_CORNERS.png)

## L2-DD individual

### Pseudoseção observada
![L2-DD pseudoseção](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_CONTINUACAO_2026-10-30/L2_DD/REAL/01_pseudosecao_observada.png)

### Malha quadrilateral
![L2-DD malha](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_CONTINUACAO_2026-10-30/L2_DD/REAL/02_malha_quadrilateral.png)

### Modelo real — Cut Corners
![L2-DD real](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_CONTINUACAO_2026-10-30/L2_DD/REAL/03_modelo_resistividade_cut_corners.png)

### Modelo sintético verdadeiro
![L2-DD verdadeiro](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_CONTINUACAO_2026-10-30/L2_DD/SINTETICO/01_modelo_verdadeiro_cut_corners.png)

### Pseudoseção sintética
![L2-DD pseudoseção sintética](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_CONTINUACAO_2026-10-30/L2_DD/SINTETICO/02_pseudosecao_sintetica.png)

### Recuperação sintética
![L2-DD recuperação](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_CONTINUACAO_2026-10-30/L2_DD/SINTETICO/03_recuperacao_sintetica_cut_corners.png)

### Comparação
![L2-DD comparação](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_CONTINUACAO_2026-10-30/L2_DD/COMPARACAO_VERDADEIRO_SINTETICO_REAL_CUT_CORNERS.png)

## L2-WS individual

### Pseudoseção observada
![L2-WS pseudoseção](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_CONTINUACAO_2026-10-30/L2_WS/REAL/01_pseudosecao_observada.png)

### Malha quadrilateral
![L2-WS malha](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_CONTINUACAO_2026-10-30/L2_WS/REAL/02_malha_quadrilateral.png)

### Modelo real — Cut Corners
![L2-WS real](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_CONTINUACAO_2026-10-30/L2_WS/REAL/03_modelo_resistividade_cut_corners.png)

### Modelo sintético verdadeiro
![L2-WS verdadeiro](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_CONTINUACAO_2026-10-30/L2_WS/SINTETICO/01_modelo_verdadeiro_cut_corners.png)

### Pseudoseção sintética
![L2-WS pseudoseção sintética](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_CONTINUACAO_2026-10-30/L2_WS/SINTETICO/02_pseudosecao_sintetica.png)

### Recuperação sintética
![L2-WS recuperação](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_CONTINUACAO_2026-10-30/L2_WS/SINTETICO/03_recuperacao_sintetica_cut_corners.png)

### Comparação
![L2-WS comparação](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_CONTINUACAO_2026-10-30/L2_WS/COMPARACAO_VERDADEIRO_SINTETICO_REAL_CUT_CORNERS.png)

## L2-PD individual — eletrodo 67 no infinito

### Pseudoseção observada
![L2-PD pseudoseção](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_CONTINUACAO_2026-10-30/L2_PD/REAL/01_pseudosecao_observada.png)

### Malha quadrilateral
![L2-PD malha](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_CONTINUACAO_2026-10-30/L2_PD/REAL/02_malha_quadrilateral.png)

### Modelo real — Cut Corners
![L2-PD real](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_CONTINUACAO_2026-10-30/L2_PD/REAL/03_modelo_resistividade_cut_corners.png)

### Modelo sintético verdadeiro
![L2-PD verdadeiro](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_CONTINUACAO_2026-10-30/L2_PD/SINTETICO/01_modelo_verdadeiro_cut_corners.png)

### Pseudoseção sintética
![L2-PD pseudoseção sintética](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_CONTINUACAO_2026-10-30/L2_PD/SINTETICO/02_pseudosecao_sintetica.png)

### Recuperação sintética
![L2-PD recuperação](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_CONTINUACAO_2026-10-30/L2_PD/SINTETICO/03_recuperacao_sintetica_cut_corners.png)

### Comparação
![L2-PD comparação](../../05_RESULTADOS_RESIPY/INVERSOES_INDIVIDUAIS_QUAD_CAVIDADE_CONTINUACAO_2026-10-30/L2_PD/COMPARACAO_VERDADEIRO_SINTETICO_REAL_CUT_CORNERS.png)

## L1 combinado — DD+WS

### Pseudoseção observada combinada
![L1 combinado pseudoseção](../../05_RESULTADOS_RESIPY/INVERSOES_COMBINADAS_QUAD_CAVIDADE_2026-10-31/L1_DD_WS_COMBINADO/REAL/01_pseudosecao_observada_combinada.png)

### Malha quadrilateral
![L1 combinado malha](../../05_RESULTADOS_RESIPY/INVERSOES_COMBINADAS_QUAD_CAVIDADE_2026-10-31/L1_DD_WS_COMBINADO/REAL/02_malha_quadrilateral.png)

### Modelo real — Cut Corners
![L1 combinado real](../../05_RESULTADOS_RESIPY/INVERSOES_COMBINADAS_QUAD_CAVIDADE_2026-10-31/L1_DD_WS_COMBINADO/REAL/03_modelo_resistividade_cut_corners.png)

### Modelo sintético verdadeiro
![L1 combinado verdadeiro](../../05_RESULTADOS_RESIPY/INVERSOES_COMBINADAS_QUAD_CAVIDADE_2026-10-31/L1_DD_WS_COMBINADO/SINTETICO/01_modelo_verdadeiro_cut_corners.png)

### Pseudoseção sintética combinada
![L1 combinado pseudoseção sintética](../../05_RESULTADOS_RESIPY/INVERSOES_COMBINADAS_QUAD_CAVIDADE_2026-10-31/L1_DD_WS_COMBINADO/SINTETICO/02_pseudosecao_sintetica_combinada.png)

### Recuperação sintética
![L1 combinado recuperação](../../05_RESULTADOS_RESIPY/INVERSOES_COMBINADAS_QUAD_CAVIDADE_2026-10-31/L1_DD_WS_COMBINADO/SINTETICO/03_recuperacao_sintetica_cut_corners.png)

### Comparação
![L1 combinado comparação](../../05_RESULTADOS_RESIPY/INVERSOES_COMBINADAS_QUAD_CAVIDADE_2026-10-31/L1_DD_WS_COMBINADO/COMPARACAO_VERDADEIRO_SINTETICO_REAL_CUT_CORNERS.png)

## L2 combinado — DD+WS+PD com eletrodo remoto

### Pseudoseção observada combinada
![L2 combinado pseudoseção](../../05_RESULTADOS_RESIPY/INVERSOES_COMBINADAS_QUAD_CAVIDADE_2026-10-31_L2_CONTINUACAO/L2_DD_WS_PD_COMBINADO/REAL/01_pseudosecao_observada_combinada.png)

### Malha quadrilateral
![L2 combinado malha](../../05_RESULTADOS_RESIPY/INVERSOES_COMBINADAS_QUAD_CAVIDADE_2026-10-31_L2_CONTINUACAO/L2_DD_WS_PD_COMBINADO/REAL/02_malha_quadrilateral.png)

### Modelo real — Cut Corners
![L2 combinado real](../../05_RESULTADOS_RESIPY/INVERSOES_COMBINADAS_QUAD_CAVIDADE_2026-10-31_L2_CONTINUACAO/L2_DD_WS_PD_COMBINADO/REAL/03_modelo_resistividade_cut_corners.png)

### Modelo sintético verdadeiro
![L2 combinado verdadeiro](../../05_RESULTADOS_RESIPY/INVERSOES_COMBINADAS_QUAD_CAVIDADE_2026-10-31_L2_CONTINUACAO/L2_DD_WS_PD_COMBINADO/SINTETICO/01_modelo_verdadeiro_cut_corners.png)

### Pseudoseção sintética combinada
![L2 combinado pseudoseção sintética](../../05_RESULTADOS_RESIPY/INVERSOES_COMBINADAS_QUAD_CAVIDADE_2026-10-31_L2_SINTETICO_FINAL/SINTETICO/02_pseudosecao_sintetica_combinada.png)

### Recuperação sintética
![L2 combinado recuperação](../../05_RESULTADOS_RESIPY/INVERSOES_COMBINADAS_QUAD_CAVIDADE_2026-10-31_L2_SINTETICO_FINAL/SINTETICO/03_recuperacao_sintetica_cut_corners.png)

### Comparação
![L2 combinado comparação](../../05_RESULTADOS_RESIPY/INVERSOES_COMBINADAS_QUAD_CAVIDADE_2026-10-31_L2_SINTETICO_FINAL/COMPARACAO_VERDADEIRO_SINTETICO_REAL_CUT_CORNERS.png)

# Síntese quantitativa

| Caso | Medições | RMS real | RMS sintético | IoU sintético | Mediana real dentro/fora (Ω·m) |
|---|---:|---:|---:|---:|---:|
| L1-DD | 651 | 1,00 | 1,00 | 0,900 | 2.852 / 3.783 |
| L1-WS | 801 | 1,22 | 1,07 | 0,900 | 3.641 / 4.770 |
| L2-DD | 819 | 1,00 | 1,26 | 0,806 | 13.727 / 2.480 |
| L2-WS | 895 | 1,07 | 1,00 | 0,800 | 17.864 / 2.435 |
| L2-PD | 1.304 | 1,08 | 1,19 | 0,848 | 14.492 / 2.850 |
| L1 DD+WS | 1.452 | 1,31 | 1,12 | 0,921 | 3.817 / 4.522 |
| L2 DD+WS+PD | 3.018 | 1,08 | 1,00 | 0,859 | 15.981 / 2.688 |

A recuperação sintética é boa em todos os casos. L1 não apresenta contraste resistivo real coincidente com a cavidade modelada; L2 apresenta anomalia resistiva forte e consistente, mas a interpretação como vazio aberto exige controle geológico ou investigação direta.

Relatório completo: [`05_RESULTADOS_RESIPY/RELATORIO_CONSOLIDADO_INVERSOES_ERT.md`](../../05_RESULTADOS_RESIPY/RELATORIO_CONSOLIDADO_INVERSOES_ERT.md).